# Notebook 3 — Advanced GANs: Pix2Pix, CycleGAN, and Evaluation Metrics

**Phase 3** deliverable. Learning objectives:
- Pix2Pix for paired image-to-image translation
- CycleGAN for unpaired style transfer
- FID and Inception Score evaluation

Complete the TODOs in `models/advanced/`, the Phase 3 losses in
`training/losses.py`, and `evaluation/metrics.py` first. Run
`pytest -m advanced -v` and `pytest -m evaluation -v` to check your progress.


In [3]:
import sys
from pathlib import Path

# Make `generative_art_studio` importable without `pip install -e .`
# REPO_ROOT = Path.cwd().parent if (Path.cwd() / "notebooks").exists() else Path.cwd()
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import torch
from generative_art_studio.utils import set_seed, plot_image_grid
from generative_art_studio.config import DEVICE

set_seed(42)
print(f"Using device: {DEVICE}")


Using device: cuda


## 1. Pix2Pix

Implement `UNetUp.forward`'s skip connection in `models/advanced/pix2pix.py`
and `pix2pix_generator_loss` in `training/losses.py`.


In [4]:
from generative_art_studio.models.advanced import UNetGenerator, PatchGANDiscriminator
from generative_art_studio.data import SyntheticImageDataset, get_dataloader

dataset = SyntheticImageDataset(num_samples=128, image_size=64)
dataloader = get_dataloader(dataset, batch_size=16)

unet = UNetGenerator(features=32).to(DEVICE)
patch_disc = PatchGANDiscriminator(in_channels=6).to(DEVICE)

images, _ = next(iter(dataloader))
images = images.to(DEVICE)
fake = unet(images)  # here input == "source" domain image; swap in a paired dataset for real use
print("Generated:", fake.shape)
plot_image_grid(fake.detach().cpu()[:8], nrow=4, title="Untrained Pix2Pix output")


Generated: torch.Size([16, 3, 64, 64])


<Figure size 400x400 with 1 Axes>

### TODO: Pix2Pix training loop

For a *paired* dataset (source_img, target_img), each step:
1. `fake = unet(source_img)`
2. Discriminator step: `patchgan_discriminator_loss(patch_disc(source_img, target_img), patch_disc(source_img, fake.detach()))`
3. Generator step: `pix2pix_generator_loss(patch_disc(source_img, fake), fake, target_img, lambda_l1=100.0)`


In [5]:
# TODO: your Pix2Pix training loop here (needs a paired dataset — see docs/PROJECT_BRIEF.md's WikiArt / Pix2Pix notes)
# TODO: Pix2Pix training loop
from torch.optim import Adam
from generative_art_studio.training.losses import (
    patchgan_discriminator_loss,
    pix2pix_generator_loss,
)

optimizer_g = Adam(unet.parameters(), lr=2e-4, betas=(0.5, 0.999))
optimizer_d = Adam(patch_disc.parameters(), lr=2e-4, betas=(0.5, 0.999))

epochs = 3

for epoch in range(epochs):
    # Build a simple paired loader from the existing synthetic dataset.
    # Each batch is paired with itself (source, target) so the Pix2Pix training loop
    # can run even when a true paired dataset is not yet defined.
    paired_dataloader = [(images, images.clone()) for images, _ in dataloader]

    for source_img, target_img in paired_dataloader:
        source_img = source_img.to(DEVICE)
        target_img = target_img.to(DEVICE)

        # ---------------------
        # Generator
        # ---------------------
        fake_img = unet(source_img)

        # ---------------------
        # Discriminator
        # ---------------------
        optimizer_d.zero_grad()

        real_pred = patch_disc(source_img, target_img)
        fake_pred = patch_disc(source_img, fake_img.detach())

        d_loss = patchgan_discriminator_loss(real_pred, fake_pred)
        d_loss.backward()
        optimizer_d.step()

        # ---------------------
        # Generator
        # ---------------------
        optimizer_g.zero_grad()

        fake_pred = patch_disc(source_img, fake_img)

        g_loss = pix2pix_generator_loss(
            fake_pred,
            fake_img,
            target_img,
            lambda_l1=100.0,
        )

        g_loss.backward()
        optimizer_g.step()

    print(
        f"Epoch [{epoch + 1}/{epochs}] "
        f"D Loss: {d_loss.item():.4f} "
        f"G Loss: {g_loss.item():.4f}"
    )

Epoch [1/3] D Loss: 0.5939 G Loss: 58.2685
Epoch [2/3] D Loss: 0.2828 G Loss: 54.9167
Epoch [3/3] D Loss: 0.2860 G Loss: 50.8556


## 2. CycleGAN

Implement `ResidualBlock.forward`'s skip connection in
`models/advanced/cyclegan.py` and `cycle_consistency_loss` /
`identity_loss` in `training/losses.py`.


In [6]:
from generative_art_studio.models.advanced import CycleGANGenerator

g_a2b = CycleGANGenerator(features=32, num_residual_blocks=4).to(DEVICE)
g_b2a = CycleGANGenerator(features=32, num_residual_blocks=4).to(DEVICE)
d_a = PatchGANDiscriminator(in_channels=3).to(DEVICE)  # unconditional: single image in
d_b = PatchGANDiscriminator(in_channels=3).to(DEVICE)

fake_b = g_a2b(images)
reconstructed_a = g_b2a(fake_b)
print("Cycle shapes:", images.shape, "->", fake_b.shape, "->", reconstructed_a.shape)


Cycle shapes: torch.Size([16, 3, 64, 64]) -> torch.Size([16, 3, 64, 64]) -> torch.Size([16, 3, 64, 64])


### TODO: CycleGAN training loop

For *unpaired* domain-A and domain-B images each step, combine:
- adversarial loss for both directions (`patchgan_discriminator_loss` / same-style generator adv term)
- `cycle_consistency_loss(real_a, g_b2a(g_a2b(real_a)))` (and the B->A->B direction)
- `identity_loss(real_b, g_a2b(real_b))` (and the mirror direction)


In [11]:
from generative_art_studio.data import SyntheticImageDataset, get_dataloader

# Unpaired Domain A dataset
dataset_a = SyntheticImageDataset(
    num_samples=128,
    image_size=64,
)

# Unpaired Domain B dataset
dataset_b = SyntheticImageDataset(
    num_samples=128,
    image_size=64,
)

# Separate dataloaders for the two domains
dataloader_a = get_dataloader(
    dataset_a,
    batch_size=16,
)

dataloader_b = get_dataloader(
    dataset_b,
    batch_size=16,
)

print("Domain A batches:", len(dataloader_a))
print("Domain B batches:", len(dataloader_b))

Domain A batches: 8
Domain B batches: 8


In [12]:
# TODO: your CycleGAN training loop here
from torch.optim import Adam
import torch.nn.functional as F

from generative_art_studio.training.losses import (
    patchgan_discriminator_loss,
    cycle_consistency_loss,
    identity_loss,
)

# Optimizers
optimizer_g = Adam(
    list(g_a2b.parameters()) + list(g_b2a.parameters()),
    lr=2e-4,
    betas=(0.5, 0.999),
)

optimizer_d_a = Adam(
    d_a.parameters(),
    lr=2e-4,
    betas=(0.5, 0.999),
)

optimizer_d_b = Adam(
    d_b.parameters(),
    lr=2e-4,
    betas=(0.5, 0.999),
)

# Loss weights
lambda_cycle = 10.0
lambda_identity = 5.0

epochs = 3

for epoch in range(epochs):

    for (real_a, _), (real_b, _) in zip(dataloader_a, dataloader_b):

        real_a = real_a.to(DEVICE)
        real_b = real_b.to(DEVICE)

        # ==========================================================
        # 1. Generate fake images
        # ==========================================================
        fake_b = g_a2b(real_a)   # A -> B
        fake_a = g_b2a(real_b)   # B -> A

        # ==========================================================
        # 2. Cycle reconstruction
        # ==========================================================
        reconstructed_a = g_b2a(fake_b)   # A -> B -> A
        reconstructed_b = g_a2b(fake_a)   # B -> A -> B

        # ==========================================================
        # 3. Identity mapping
        # ==========================================================
        identity_b = g_a2b(real_b)        # B -> B
        identity_a = g_b2a(real_a)        # A -> A

        # ==========================================================
        # 4. Generator update
        # ==========================================================
        optimizer_g.zero_grad()

        # Adversarial loss: generators want fake images classified as real
        fake_b_pred = d_b(fake_b)
        fake_a_pred = d_a(fake_a)

        adv_a2b = F.binary_cross_entropy_with_logits(
            fake_b_pred,
            torch.ones_like(fake_b_pred),
        )

        adv_b2a = F.binary_cross_entropy_with_logits(
            fake_a_pred,
            torch.ones_like(fake_a_pred),
        )

        # Cycle-consistency losses
        cycle_a = cycle_consistency_loss(
            real_a,
            reconstructed_a,
        )

        cycle_b = cycle_consistency_loss(
            real_b,
            reconstructed_b,
        )

        # Identity losses
        identity_a_loss = identity_loss(
            real_a,
            identity_a,
        )

        identity_b_loss = identity_loss(
            real_b,
            identity_b,
        )

        # Total generator loss
        g_loss = (
            adv_a2b
            + adv_b2a
            + lambda_cycle * (cycle_a + cycle_b)
            + lambda_identity * (identity_a_loss + identity_b_loss)
        )

        g_loss.backward()
        optimizer_g.step()

        # ==========================================================
        # 5. Discriminator A update
        # ==========================================================
        optimizer_d_a.zero_grad()

        real_a_pred = d_a(real_a)
        fake_a_pred = d_a(fake_a.detach())

        d_a_loss = patchgan_discriminator_loss(
            real_a_pred,
            fake_a_pred,
        )

        d_a_loss.backward()
        optimizer_d_a.step()

        # ==========================================================
        # 6. Discriminator B update
        # ==========================================================
        optimizer_d_b.zero_grad()

        real_b_pred = d_b(real_b)
        fake_b_pred = d_b(fake_b.detach())

        d_b_loss = patchgan_discriminator_loss(
            real_b_pred,
            fake_b_pred,
        )

        d_b_loss.backward()
        optimizer_d_b.step()

    print(
        f"Epoch [{epoch + 1}/{epochs}] "
        f"G Loss: {g_loss.item():.4f} | "
        f"D_A Loss: {d_a_loss.item():.4f} | "
        f"D_B Loss: {d_b_loss.item():.4f}"
    )
  

Epoch [1/3] G Loss: 17.5662 | D_A Loss: 0.6880 | D_B Loss: 0.6513
Epoch [2/3] G Loss: 17.6189 | D_A Loss: 0.5657 | D_B Loss: 0.5395
Epoch [3/3] G Loss: 18.4538 | D_A Loss: 0.4258 | D_B Loss: 0.3862


## 3. Evaluation: FID & Inception Score

In [13]:
from generative_art_studio.evaluation.metrics import compute_fid, compute_inception_score
import numpy as np

# Sanity check with synthetic features first (this is exactly what the graded tests do):
rng = np.random.default_rng(0)
real_features = rng.normal(size=(200, 32))
fake_features_good = real_features + rng.normal(scale=0.1, size=(200, 32))  # close to real
fake_features_bad = rng.normal(loc=5.0, size=(200, 32))  # far from real

print("FID (good generator):", compute_fid(real_features, fake_features_good))
print("FID (bad generator): ", compute_fid(real_features, fake_features_bad))


FID (good generator): 0.02841364788912705
FID (bad generator):  808.5358754507192


### TODO: real FID/IS on your trained models

```python
from generative_art_studio.evaluation.metrics import get_inception_feature_extractor

extractor = get_inception_feature_extractor().to(DEVICE)  # downloads pretrained weights
real_feats, _ = extractor(real_batch.to(DEVICE))
fake_feats, fake_probs = extractor(generated_batch.to(DEVICE))

fid = compute_fid(real_feats.cpu().numpy(), fake_feats.cpu().numpy())
is_mean, is_std = compute_inception_score(fake_probs.cpu().numpy())
```

Run this for at least two of your trained generators and compare.


In [16]:
# TODO: compute real FID/IS for your trained models here

from generative_art_studio.evaluation.metrics import (
    get_inception_feature_extractor,
    compute_fid,
    compute_inception_score,
)

# Get a real batch
real_batch, _ = next(iter(dataloader))
real_batch = real_batch.to(DEVICE)

# Generate images from your trained generators
with torch.no_grad():
    generated_batch_1 = g_a2b(real_batch)
    generated_batch_2 = g_b2a(real_batch)

# Load pretrained Inception v3
extractor = get_inception_feature_extractor().to(DEVICE)
extractor.eval()

# Extract features and probabilities
with torch.no_grad():
    real_feats, real_probs = extractor(real_batch)

    fake_feats_1, fake_probs_1 = extractor(generated_batch_1)
    fake_feats_2, fake_probs_2 = extractor(generated_batch_2)

# Convert to NumPy
real_feats_np = real_feats.cpu().numpy()
fake_feats_1_np = fake_feats_1.cpu().numpy()
fake_feats_2_np = fake_feats_2.cpu().numpy()

fake_probs_1_np = fake_probs_1.cpu().numpy()
fake_probs_2_np = fake_probs_2.cpu().numpy()

# ==========================================================
# FID
# ==========================================================

fid_1 = compute_fid(
    real_feats_np,
    fake_feats_1_np,
)

fid_2 = compute_fid(
    real_feats_np,
    fake_feats_2_np,
)

# ==========================================================
# Inception Score
# ==========================================================

is_mean_1, is_std_1 = compute_inception_score(
    fake_probs_1_np,
)

is_mean_2, is_std_2 = compute_inception_score(
    fake_probs_2_np,
)

# ==========================================================
# Results
# ==========================================================

print("Generator 1 — G_A2B")
print(f"FID: {fid_1:.4f}")
print(f"Inception Score: {is_mean_1:.4f} ± {is_std_1:.4f}")

print("\nGenerator 2 — G_B2A")
print(f"FID: {fid_2:.4f}")
print(f"Inception Score: {is_mean_2:.4f} ± {is_std_2:.4f}")
  

Generator 1 — G_A2B
FID: 33.5532
Inception Score: 1.0058 ± 0.0060

Generator 2 — G_B2A
FID: 45.9206
Inception Score: 1.0045 ± 0.0039


## Reflection (for your Technical Report)

- Compare FID/IS across your models in one table.
- Where do FID/IS agree or disagree with your own visual judgment of
  quality? What might explain a disagreement?
